# **Week 7 Assignment: Retrieval-Augmented Generation (RAG) System**

# Step 1: Import Libraries

In [86]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu pypdf transformers sentence-transformers accelerate

In [87]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from transformers import pipeline

#Step 2. Document Ingestion

The first stage of the RAG pipeline is document ingestion. The system accepts custom documents such as PDF files, text files, or domain-specific datasets as input. These documents are loaded and converted into raw text using appropriate document loaders. The extracted text serves as the knowledge base for the question-answering system.

In [88]:
from google.colab import files

uploaded = files.upload()

Saving What is Retrieval.pdf to What is Retrieval (2).pdf


In [89]:
file_path = "/content/What is Retrieval.pdf"

if file_path.endswith(".pdf"):
    loader = PyPDFLoader(file_path)

elif file_path.endswith(".txt"):
    loader = TextLoader(file_path)

documents = loader.load()

print("Total Pages:", len(documents))

Total Pages: 5


# Step 3: Text Chunking

The extracted text is divided into smaller, manageable chunks using a text-splitting strategy such as RecursiveCharacterTextSplitter. Chunking improves retrieval efficiency by ensuring that each text segment contains a focused piece of information while maintaining contextual continuity through overlapping regions.

In [90]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 20


# Step 4: Text Embedding

Each text chunk is converted into a numerical vector representation using a pre-trained embedding model. These embeddings capture the semantic meaning of the text, enabling the system to compare the similarity between user queries and stored document chunks. A commonly used embedding model is sentence-transformers/all-MiniLM-L6-v2, which generates 384-dimensional embeddings.

In [91]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Step 5: Vector Database

The generated embeddings are stored in a vector database such as FAISS. The vector database enables efficient similarity search by indexing the embedding vectors, allowing the system to retrieve the most relevant document chunks in response to a user query.

In [92]:
vector_db = FAISS.from_documents(
    chunks,
    embeddings
)

print("Vector Database Created Successfully")

Vector Database Created Successfully


# Step 6: Load LLM
Objective

Load a pre-trained Large Language Model (LLM) that generates answers based on the retrieved context and the user's query.

Implementation
The Hugging Face pipeline is used to load the TinyLlama-1.1B-Chat model.
This instruction-tuned model understands prompts and generates human-like responses.
The model receives the retrieved context and the user's question as input and produces the final answer

In [107]:
from transformers import pipeline

generator = pipeline(
    task="text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device_map="auto"
)

print("LLM Loaded Successfully")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LLM Loaded Successfully


# Step 7: User Query Processing

When a user submits a question, the system converts the query into an embedding using the same embedding model that was used for the document chunks. Using the same embedding space ensures meaningful similarity comparisons between the query and stored document vectors.

In [118]:
query = input("Ask your question: ")

Ask your question: working


# Step 8: Context Retrieval

The vector database performs a similarity search between the query embedding and stored document embeddings. Based on similarity scores, the system retrieves the top-k most relevant text chunks. These retrieved chunks provide the contextual information required for generating an accurate answer.

In [119]:
docs = vector_db.similarity_search(
    query,
    k=8
)

print("\nRetrieved Documents\n")

for i, doc in enumerate(docs, start=1):
    print("="*80)
    print(f"Chunk {i}")
    print("="*80)
    print(doc.page_content)
    print()


Retrieved Documents

Chunk 1
time or scheduled so the system always retrieves latest information. 
What Problems does RAG solve 
1. Hallucinations: Traditional generative models can produce incorrect information. 
RAG reduces this risk by retrieving verified, external data to ground responses in 
factual knowledge.

Chunk 2
the model’s effectiveness. 
4. Bias and Fairness: It can inherit biases present in the training data or retrieved 
documents, necessitating ongoing efforts to ensure fairness and mitigate biases. 
RAG Applications 
1. Question-Answering Systems: It enables chatbots or virtual assistants to pull 
information from a knowledge base or documents and generate accurate, context 
aware answers.

Chunk 3
4. Information Retrieval: Goes beyond traditional search by retrieving documents and 
generating meaningful summaries of their content.

Chunk 4
costs and computational load. 
6. Scalability Across Domains: It is adaptable to diverse industries from healthcare to 
finance 

# Step 9: Create Context
Objective

The retrieved document chunks are combined into a single context that will be provided to the Language Model (LLM). This context contains the most relevant information related to the user's query.

Implementation
The retrieved chunks are concatenated into one text block.
Double newline characters (\n\n) are used to separate each chunk for better readability.
The generated context acts as the knowledge source for answer generation.

In [120]:
context = "\n\n".join(
    [doc.page_content for doc in docs]
)

# Step 10: Create Prompt
Objective

Construct a prompt that instructs the Language Model to answer the user's question using only the retrieved context.

Implementation
The retrieved context is inserted into the prompt.
The user's question is appended after the context.
The prompt explicitly instructs the model to answer only from the provided context.
If the required information is not available in the context, the model is instructed to respond that the answer is not found.

In [121]:
prompt = f"""
You are a helpful assistant.

Use ONLY the information provided in the context.

If the answer is not present in the context, reply exactly:

Answer not found in the document.

Context:
{context}

Question:
{query}

Give a complete, detailed answer.
"""

# Step 11: Answer Generation

The retrieved document chunks are combined with the user's question to create a prompt for the language model. The language model uses this contextual information to generate an answer that is grounded in the retrieved documents rather than relying solely on its pre-trained knowledge. This approach improves factual accuracy and reduces hallucinations.

In [122]:
response = generator(
    prompt,
    max_new_tokens=300,
    do_sample=False,
    temperature=0.1,
    top_p=0.95,
    eos_token_id=generator.tokenizer.eos_token_id,
    return_full_text=False
)

print("\nGenerated Answer:\n")
print(response[0]["generated_text"])

[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Answer:


Answer:
RAG is a powerful tool for generating accurate and context-aware responses. It combines 
retrieval and generation to use external data for more factual and context-aware responses. 
RAG can be used in various applications, including question-answering systems, information 
retrieval, and chatbots. The model's effectiveness depends on the quality of the retrieved 
documents, which can be improved by augmenting the LLM prompt. The model's complexity is 
addressed by careful tuning and optimization, and the latency is mitigated by combining 
retrieval and generation. The model's accuracy is enhanced by incorporating external data 
and embeddings, which are refreshed regularly. The model's output is context-aware and 
grounded in reliable data, making interactions more informative and personalized.


# Conclusion

The Retrieval-Augmented Generation (RAG) system combines information retrieval with language generation to provide accurate, context-aware answers from custom documents. The pipeline includes document ingestion, text preprocessing, chunking, embedding generation, vector storage, similarity-based retrieval, and language model inference. By grounding responses in retrieved document content, the system significantly improves factual accuracy and is suitable for applications such as document question answering, enterprise knowledge management, intelligent search systems, research assistants, and AI-powered chatbots.